# Hierarchical Planning AI System using CrewAI

## Overview
This notebook implements a comprehensive Hierarchical Planning AI System using CrewAI that breaks down a high-level event goal into a detailed, multi-level event plan. The system orchestrates 5 specialized agents that work together to research vendors, build budgets, design schedules, and produce a final consolidated event plan.

## Key Features
- **5 Specialized Agents**: Event Planner, Vendor Research, Budget Analyst, Schedule Designer, Final Event Coordinator
- **Dynamic Inputs**: event_goal, date, attendee_count, preferences
- **Hierarchical Planning**: Multi-level breakdown (Level 1 → Level 2 → Level 3)
- **Comprehensive Output**: Plan, research, budget, schedule, and final consolidated deliverable

In [2]:
# Section 1: Import Required Libraries and Initialize LLM
import os
from dotenv import load_dotenv
from crewai import Agent, Task, Crew

# Load environment variables
load_dotenv()

# Initialize OpenAI API key
openai_api_key = os.getenv("OPENAI_API_KEY")
if not openai_api_key:
    raise ValueError("OPENAI_API_KEY environment variable not set. Please set it in your .env file.")

# Set API key as environment variable for crewai to use
print(openai_api_key)
os.environ["OPENAI_API_KEY"] = openai_api_key

# Model name to use
model_name = "gpt-3.5-turbo"

print("✓ Libraries imported and LLM configured successfully")

voc-43036939317535049919856a489f8ae6c6d7.98676241
✓ Libraries imported and LLM configured successfully


## Section 2: Define Event Planner Agent (Hierarchical Planner)

In [3]:
# Task A: Define all 5 Agents

# Agent 1: Event Planner Agent (Hierarchical Planner)
event_planner_agent = Agent(
    role="Strategic Event Planner",
    goal="""Break the high-level event goal into a structured, hierarchical plan with multiple levels.
    Identify key pillars: venue, logistics, speakers, vendors, schedule, and budget.
    Generate sub-tasks and success metrics for each pillar.
    Produce a clear 3-level hierarchical breakdown.""",
    backstory="""You are an expert event strategist with 20+ years of experience in planning large-scale corporate events.
    You excel at breaking complex requirements into manageable, hierarchical plans.
    You understand dependencies between different planning pillars and create realistic timelines.
    Your plans are always actionable and clearly communicated.""",
    verbose=True,
    allow_delegation=False,
    model=model_name
)

print("✓ Event Planner Agent created")

✓ Event Planner Agent created


## Section 3: Define Vendor Research Agent

In [4]:
# Agent 2: Vendor Research Agent
vendor_research_agent = Agent(
    role="Vendor & Venue Research Specialist",
    goal="""Conduct thorough research into event vendors and venues.
    Identify and evaluate catering vendors (2-3 options), audio/visual providers, 
    seating and event décor companies, and stage setup providers.
    Provide detailed comparisons with pros, cons, pricing, and recommendations.""",
    backstory="""You are a meticulous research specialist with deep knowledge of the event industry.
    You have extensive contacts with vendors across multiple categories.
    You excel at comparing vendors objectively, considering price, quality, reputation, and fit.
    You always provide multiple options and detailed analysis to support decision-making.""",
    verbose=True,
    allow_delegation=False,
    model=model_name
)

print("✓ Vendor Research Agent created")

✓ Vendor Research Agent created


## Section 4: Define Budget Analyst Agent

In [5]:
# Agent 3: Budget Analyst Agent
budget_analyst_agent = Agent(
    role="Event Budget Manager",
    goal="""Build a detailed, itemized budget that covers all event expenses.
    Break down costs by category: venue, food & beverage, décor, A/V, labour, marketing.
    Identify must-book-first items and contingency costs.
    Suggest cost-saving strategies without compromising event quality.""",
    backstory="""You are a financial expert specializing in event budget management.
    You have managed budgets ranging from $10,000 to $10M+ for diverse events.
    You understand vendor pricing, negotiate effectively, and identify cost-saving opportunities.
    You always build in contingency buffers and track ROI metrics.""",
    verbose=True,
    allow_delegation=False,
    model=model_name
)

print("✓ Budget Analyst Agent created")

✓ Budget Analyst Agent created


## Section 5: Define Schedule Designer Agent

In [6]:
# Agent 4: Schedule Designer Agent
schedule_designer_agent = Agent(
    role="Event Day Coordinator",
    goal="""Design a detailed, minute-by-minute event day schedule.
    Include keynote sessions, buffer times, breaks, workshops, and networking blocks.
    Plan speaker preparation windows and technical setup requirements.
    Ensure realistic timing and smooth flow throughout the day.""",
    backstory="""You are a logistics expert with 15+ years of event execution experience.
    You've coordinated thousands of hours of events, from intimate conferences to massive summits.
    You understand timing, crowd flow, technical requirements, and contingency planning.
    Your schedules are always realistic, flexible, and designed for optimal guest experience.""",
    verbose=True,
    allow_delegation=False,
    model=model_name
)

print("✓ Schedule Designer Agent created")

✓ Schedule Designer Agent created


## Section 6: Define Final Event Coordinator Agent

In [7]:
# Agent 5: Final Event Coordinator Agent
final_coordinator_agent = Agent(
    role="Chief Event Coordinator",
    goal="""Consolidate all planning elements into one comprehensive, structured event plan.
    Integrate hierarchical plan, vendor research, budget, and schedule into a unified roadmap.
    Include booking checklist, timeline, risk assessment, and contingency procedures.
    Produce a final, actionable deliverable ready for execution.""",
    backstory="""You are the ultimate event orchestrator and integrator with 25+ years of experience.
    You excel at synthesizing complex information from multiple sources into actionable plans.
    You've led teams through successful execution of high-stakes events worth millions of dollars.
    Your final plans are comprehensive, reviewed for consistency, and include every detail needed for success.""",
    verbose=True,
    allow_delegation=True,  # Can delegate to other agents if needed
    model=model_name
)

print("✓ Final Event Coordinator Agent created")
print("\n✓ All 5 agents successfully created!")

✓ Final Event Coordinator Agent created

✓ All 5 agents successfully created!


## Section 7: Create Tasks for Each Agent

Task B: Define one CrewAI Task for each agent with structured descriptions and expected outputs. Dynamic placeholders will be injected into task descriptions.

In [8]:
# Task B: Create 5 Tasks for each agent

def create_event_planning_tasks(event_goal, date, attendee_count, preferences):
    """
    Create a set of tasks with dynamic inputs for hierarchical event planning.
    
    Args:
        event_goal: The high-level event goal
        date: Event date
        attendee_count: Expected attendees
        preferences: Special preferences and constraints
    
    Returns:
        List of Task objects
    """
    
    # Task 1: Event Planning Task
    task_1_hierarchical_plan = Task(
        description=f"""
        Analyze and break down this event requirement into a hierarchical plan:
        
        EVENT GOAL: {event_goal}
        DATE: {date}
        ATTENDEE COUNT: {attendee_count}
        PREFERENCES: {preferences}
        
        Create a structured 3-level hierarchical breakdown:
        
        LEVEL 1 (Strategic Pillars):
        - Venue & Logistics
        - Speakers & Content
        - Vendors & Services
        - Schedule & Timeline
        - Budget & Resources
        
        LEVEL 2 (Sub-components):
        For each Level 1 pillar, identify 3-4 key sub-components
        
        LEVEL 3 (Actionable Tasks):
        For each Level 2 component, identify 2-3 specific, actionable tasks
        
        Include:
        - Dependencies between pillars
        - Success metrics for each pillar
        - Key milestones and deadlines
        - Resource requirements
        """,
        agent=event_planner_agent,
        expected_output="""A comprehensive 3-level hierarchical event plan including:
        - Level 1: 5 strategic pillars
        - Level 2: Sub-components for each pillar
        - Level 3: Specific actionable tasks
        - Dependencies and success metrics"""
    )
    
    # Task 2: Vendor Research Task
    task_2_vendor_research = Task(
        description=f"""
        Research and recommend vendors for this event:
        
        EVENT GOAL: {event_goal}
        DATE: {date}
        ATTENDEE COUNT: {attendee_count}
        PREFERENCES: {preferences}
        
        Research and provide options for:
        
        1. CATERING VENDORS (provide 2-3 options):
           - Estimated cost per person
           - Menu options (including dietary requirements)
           - Service level and experience
           - Reviews/references
        
        2. AUDIO/VISUAL PROVIDERS (2-3 options):
           - Equipment available
           - Technical team capabilities
           - Pricing
           - Backup systems
        
        3. SEATING & DÉCOR PROVIDERS (2-3 options):
           - Styling options matching specified preferences
           - Capacity and flexibility
           - Installation/cleanup services
           - Pricing
        
        4. VENUE OPTIONS (3-5 options if not specified):
           - Capacity and layout flexibility
           - Technical infrastructure
           - Parking and accessibility
           - Estimated rental cost
        
        For each vendor option, provide:
        - Pros and cons
        - Suitability rating for this event
        - Availability on the specified date
        - Key contact information
        """,
        agent=vendor_research_agent,
        expected_output="""Detailed vendor research summary including:
        - 2-3 catering options with pricing and menus
        - 2-3 A/V vendor options with capabilities
        - 2-3 décor provider options
        - 3-5 venue alternatives with capacity/pricing
        - Pros, cons, and suitability ratings for each"""
    )
    
    # Task 3: Budget Analysis Task
    task_3_budget = Task(
        description=f"""
        Build a comprehensive budget for this event:
        
        EVENT GOAL: {event_goal}
        DATE: {date}
        ATTENDEE COUNT: {attendee_count}
        PREFERENCES: {preferences}
        
        Create a detailed, itemized budget including:
        
        1. VENUE:
           - Rental cost
           - Setup/teardown fees
           - Equipment rental if needed
        
        2. FOOD & BEVERAGE:
           - Meals (breakfast, lunch, dinner)
           - Beverages and refreshments
           - Dietary special requirements
        
        3. DÉCOR & SETUP:
           - Decoration based on preferences
           - Table settings and linens
           - Lighting and ambiance
        
        4. AUDIO/VISUAL & TECH:
           - AV equipment rental
           - Microphones, speakers, projectors
           - Live streaming setup (if needed)
        
        5. STAFFING & LABOR:
           - Event coordinator
           - Setup/breakdown crew
           - Day-of staff
        
        6. MARKETING & INVITATIONS:
           - Invitations and printing
           - Marketing materials
           - Signage
        
        7. CONTINGENCY (10-15%):
           - Emergency funds
           - Unexpected costs
        
        For each category provide:
        - Line items
        - Unit costs
        - Quantities
        - Subtotals
        - % of total budget
        
        Include:
        - Total estimated budget
        - Cost per attendee
        - "Must-book-first" items (those with earliest deadlines)
        - Cost optimization suggestions
        """,
        agent=budget_analyst_agent,
        expected_output="""Comprehensive budget breakdown including:
        - Itemized costs for all 7 categories
        - Total estimated budget
        - Cost per attendee
        - Budget allocation percentages
        - "Must-book-first" items
        - Cost-saving recommendations"""
    )
    
    # Task 4: Schedule Design Task
    task_4_schedule = Task(
        description=f"""
        Design a detailed event day schedule for:
        
        EVENT GOAL: {event_goal}
        DATE: {date}
        ATTENDEE COUNT: {attendee_count}
        PREFERENCES: {preferences}
        
        Create a comprehensive day schedule including:
        
        PRE-EVENT (hours before):
        - Venue setup and testing
        - Staff arrival and briefing
        - Technical rehearsal
        
        GUEST EXPERIENCE:
        - Guest arrival and check-in (with buffer)
        - Networking/refreshments
        - Keynote session(s)
        - Interactive workshops or sessions
        - Meal services (breakfast, breaks, lunch)
        - More sessions/activities
        - Networking time
        - Closing remarks/wrap-up
        
        TECHNICAL TIMELINE:
        - Speaker prep times
        - Technical setup windows
        - Testing and soundchecks
        
        POST-EVENT:
        - Cleanup timeline
        - Staff departure
        - Venue restoration
        
        For each time block (15-30 min intervals), include:
        - Exact time range
        - Activity description
        - Responsible party/team
        - Resources needed
        - Notes and contingencies
        
        Considerations:
        - Optimal flow and pacing
        - Staff coordination
        - Attendee experience
        - Buffer times for transitions
        - Contingency adjustments
        """,
        agent=schedule_designer_agent,
        expected_output="""Detailed minute-by-minute event schedule including:
        - Pre-event timeline (setup, rehearsal)
        - Full guest experience (arrival to departure)
        - Time-blocked activities (15-30 min blocks)
        - Technical requirements and prep times
        - Responsible parties for each block
        - Contingency procedures and buffers"""
    )
    
    # Task 5: Final Consolidation Task
    task_5_consolidation = Task(
        description=f"""
        Consolidate all planning elements into a final, comprehensive event plan:
        
        EVENT GOAL: {event_goal}
        DATE: {date}
        ATTENDEE COUNT: {attendee_count}
        PREFERENCES: {preferences}
        
        Review all prior planning outputs and create one integrated final plan including:
        
        1. EXECUTIVE SUMMARY:
           - Event overview
           - Goals and success criteria
           - Expected outcomes
           - Key stakeholders
        
        2. HIERARCHICAL EVENT PLAN:
           - 3-level breakdown (from Task 1)
           - Key pillars and components
           - Success metrics
        
        3. VENDOR & VENUE RECOMMENDATIONS:
           - Selected vendors with rationale
           - Backup options
           - Contact information
           - Key dates and deadlines
        
        4. COMPREHENSIVE BUDGET:
           - Itemized breakdown by category
           - Total estimated cost
           - Payment schedule
           - Contingency details
        
        5. EVENT DAY MASTER SCHEDULE:
           - Minute-by-minute timeline
           - Staff assignments
           - Technical requirements
           - Contingency procedures
        
        6. DETAILED BOOKING CHECKLIST:
           - All items to book with deadlines
           - Confirmations to obtain
           - Deposits required
           - Payment schedules
        
        7. RISK MITIGATION:
           - Potential risks
           - Mitigation strategies
           - Contingency plans
           - Emergency contacts
        
        8. EXECUTION ROADMAP:
           - Pre-event milestones
           - Day-before checklist
           - Day-of execution checklist
           - Post-event follow-up
        
        Ensure:
        - All components are consistent
        - No conflicting information
        - Clear accountability (who, what, when, where, why)
        - Realistic and actionable
        """,
        agent=final_coordinator_agent,
        expected_output="""Comprehensive final event plan document including:
        - Executive summary of event
        - Full hierarchical breakdown
        - Selected vendors with rationale
        - Complete budget with payment schedule
        - Master schedule with contingencies
        - Detailed booking checklist with deadlines
        - Risk assessment and mitigation
        - Complete execution roadmap"""
    )
    
    return [task_1_hierarchical_plan, task_2_vendor_research, task_3_budget, 
            task_4_schedule, task_5_consolidation]

print("✓ Task creation function defined successfully")

✓ Task creation function defined successfully


## Section 8: Build and Configure the Crew

Task C: Create a CrewAI Crew instance with all five agents and their corresponding tasks.

In [9]:
def build_event_planning_crew(event_goal, date, attendee_count, preferences):
    """
    Task C: Build and configure the CrewAI Crew for event planning.
    
    Args:
        event_goal: The high-level event goal
        date: Event date
        attendee_count: Expected attendees
        preferences: Special preferences and constraints
    
    Returns:
        Configured Crew instance ready to execute
    """
    
    # Create tasks with dynamic inputs
    tasks = create_event_planning_tasks(event_goal, date, attendee_count, preferences)
    
    # Define agents list
    agents = [
        event_planner_agent,
        vendor_research_agent,
        budget_analyst_agent,
        schedule_designer_agent,
        final_coordinator_agent
    ]
    
    # Create and configure the Crew
    crew = Crew(
        agents=agents,
        tasks=tasks,
        verbose=True,
        max_iterations=4  # Maximum iterations for task execution
    )
    
    return crew

print("✓ Crew building function defined successfully")

✓ Crew building function defined successfully


## Section 9: Run the System with Dynamic Inputs

Task D: Execute the crew using crew.kickoff() with sample dynamic inputs.

In [10]:
# Define dynamic inputs for testing
sample_inputs = {
    "event_goal": "Organize a 1-day corporate innovation summit focusing on AI and automation.",
    "date": "2025-11-15",
    "attendee_count": "300 attendees",
    "preferences": "Premium decor, vegetarian+vegan meal options, mid-range budget, interactive sessions"
}

print("\n" + "="*80)
print("HIERARCHICAL EVENT PLANNING SYSTEM - EXECUTION")
print("="*80)
print("\nDynamic Inputs:")
print(f"  Event Goal: {sample_inputs['event_goal']}")
print(f"  Date: {sample_inputs['date']}")
print(f"  Attendees: {sample_inputs['attendee_count']}")
print(f"  Preferences: {sample_inputs['preferences']}")
print("\n" + "="*80)
print("Building Crew and Starting Planning Process...")
print("="*80 + "\n")

# Build the crew with dynamic inputs
crew = build_event_planning_crew(
    event_goal=sample_inputs["event_goal"],
    date=sample_inputs["date"],
    attendee_count=sample_inputs["attendee_count"],
    preferences=sample_inputs["preferences"]
)

print("\n✓ Crew built successfully!")
print("\nStarting crew execution...\n")

# Task D: Execute the crew using async kickoff with dynamic inputs
try:
    result = await crew.kickoff_async(
        inputs={
            "event_goal": sample_inputs["event_goal"],
            "date": sample_inputs["date"],
            "attendee_count": sample_inputs["attendee_count"],
            "preferences": sample_inputs["preferences"]
        }
    )
except Exception as e:
    # Handle quota and other LLM errors gracefully
    print("\n⚠️ Crew execution failed:")
    print(str(e))
    print("\nSuggestion: Check your OpenAI API key, billing, and quota. Or run the crew with smaller inputs/tests.")
    result = None

if result is not None:
    print("\n" + "="*80)
    print("CREW EXECUTION COMPLETED")
    print("="*80)
else:
    print("\nCrew did not complete due to the error above.")



HIERARCHICAL EVENT PLANNING SYSTEM - EXECUTION

Dynamic Inputs:
  Event Goal: Organize a 1-day corporate innovation summit focusing on AI and automation.
  Date: 2025-11-15
  Attendees: 300 attendees
  Preferences: Premium decor, vegetarian+vegan meal options, mid-range budget, interactive sessions

Building Crew and Starting Planning Process...


✓ Crew built successfully!

Starting crew execution...



╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.15.1                                                                                        │
│  Latest version:  1.15.2                                                                                        │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 388c1687-048e-4860-9af2-e359d759bfea                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│          Analyze and break down this event requirement into a hierarchical plan:                                │
│                                                                                                                 │
│          EVENT GOAL: Organize a 1-day corporate innovation summit focusing on AI and automation.                │
│          DATE: 2025-11-15                                                                                       │
│          ATTENDEE COUNT: 300 attendees                                                                          │
│          PREFERENCES: Premium decor, vegetarian+vegan meal options, mid-range budget, interactive sessions      │
│                                                                                                                 │
│          Create a structured 3-level hierarchical breakdown:                                                    │
│                                                                                                                 │
│          LEVEL 1 (Strategic Pillars):                                                                           │
│          - Venue & Logistics                                                                                    │
│          - Speakers & Content                                                                                   │
│          - Vendors & Services                                                                                   │
│          - Schedule & Timeline                                                                                  │
│          - Budget & Resources                                                                                   │
│                                                                                                                 │
│          LEVEL 2 (Sub-components):                                                                              │
│          For each Level 1 pillar, identify 3-4 key sub-components                                               │
│                                                                                                                 │
│          LEVEL 3 (Actionable Tasks):                                                                            │
│          For each Level 2 component, identify 2-3 specific, actionable tasks                                    │
│                                                                                                                 │
│          Include:                                                                                               │
│          - Dependencies between pillars                                                                         │
│          - Success metrics for each pillar                                                                      │
│          - Key milestones and deadlines                                                                         │
│          - Resource requirements                                                                                │
│                                                                                                                 │
│  ID: 021820f9-7326-4939-8edd-224d14652220                                                                       │
│                                                                                                                 │
│                                                        

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Strategic Event Planner                                                                                 │
│                                                                                                                 │
│  Task:                                                                                                          │
│          Analyze and break down this event requirement into a hierarchical plan:                                │
│                                                                                                                 │
│          EVENT GOAL: Organize a 1-day corporate innovation summit focusing on AI and automation.                │
│          DATE: 2025-11-15                                                                                       │
│          ATTENDEE COUNT: 300 attendees                                                                          │
│          PREFERENCES: Premium decor, vegetarian+vegan meal options, mid-range budget, interactive sessions      │
│                                                                                                                 │
│          Create a structured 3-level hierarchical breakdown:                                                    │
│                                                                                                                 │
│          LEVEL 1 (Strategic Pillars):                                                                           │
│          - Venue & Logistics                                                                                    │
│          - Speakers & Content                                                                                   │
│          - Vendors & Services                                                                                   │
│          - Schedule & Timeline                                                                                  │
│          - Budget & Resources                                                                                   │
│                                                                                                                 │
│          LEVEL 2 (Sub-components):                                                                              │
│          For each Level 1 pillar, identify 3-4 key sub-components                                               │
│                                                                                                                 │
│          LEVEL 3 (Actionable Tasks):                                                                            │
│          For each Level 2 component, identify 2-3 specific, actionable tasks                                    │
│                                                                                                                 │
│          Include:                                                                                               │
│          - Dependencies between pillars                                                                         │
│          - Success metrics for each pillar                                                                      │
│          - Key milestones and deadlines                                                                         │
│          - Resource requirements                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰────────────────────────────────────────────────────────

...

[CrewAIEventsBus] Warning: Event pairing mismatch. 'crew_kickoff_failed' closed 'agent_execution_started' (expected
'crew_kickoff_started')

╭──────────────────────────────────────────────── 📋 Task Failure ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Failed                                                                                                    │
│  Name:                                                                                                          │
│          Analyze and break down this event requirement into a hierarchical plan:                                │
│                                                                                                                 │
│          EVENT GOAL: Organize a 1-day corporate innovation summit focusing on AI and automation.                │
│          DATE: 2025-11-15                                                                                       │
│          ATTENDEE COUNT: 300 attendees                                                                          │
│          PREFERENCES: Premium decor, vegetarian+vegan meal options, mid-range budget, interactive sessions      │
│                                                                                                                 │
│          Create a structured 3-level hierarchical breakdown:                                                    │
│                                                                                                                 │
│          LEVEL 1 (Strategic Pillars):                                                                           │
│          - Venue & Logistics                                                                                    │
│          - Speakers & Content                                                                                   │
│          - Vendors & Services                                                                                   │
│          - Schedule & Timeline                                                                                  │
│          - Budget & Resources                                                                                   │
│                                                                                                                 │
│          LEVEL 2 (Sub-components):                                                                              │
│          For each Level 1 pillar, identify 3-4 key sub-components                                               │
│                                                                                                                 │
│          LEVEL 3 (Actionable Tasks):                                                                            │
│          For each Level 2 component, identify 2-3 specific, actionable tasks                                    │
│                                                                                                                 │
│          Include:                                                                                               │
│          - Dependencies between pillars                                                                         │
│          - Success metrics for each pillar                                                                      │
│          - Key milestones and deadlines                                                                         │
│          - Resource requirements                                                                                │
│                                                                                                                 │
│  Agent: Strategic Event Planner                                                                                 │
│                                                                                                                 │
│                                                        


⚠️ Crew execution failed:
Error code: 401 - {'error': {'message': 'Incorrect API key provided: voc-4303*************************************6241. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}

Suggestion: Check your OpenAI API key, billing, and quota. Or run the crew with smaller inputs/tests.

Crew did not complete due to the error above.


╭───────────────────────────────────────────────── Crew Failure ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Failed                                                                                          │
│  Name: crew                                                                                                     │
│  ID: 388c1687-048e-4860-9af2-e359d759bfea                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## Section 10: Display Consolidated Event Plan Output

Parse and display the final consolidated output with all sections.

In [11]:
# Display final consolidated output
print("\n" + "="*80)
print("FINAL CONSOLIDATED EVENT PLAN")
print("="*80 + "\n")

print(result)

print("\n" + "="*80)
print("PLANNING PROCESS COMPLETED SUCCESSFULLY")
print("="*80)

print("\n📋 OUTPUT SUMMARY:")
print("-" * 80)
print("✓ Hierarchical Event Plan: Generated")
print("✓ Vendor Research Summary: Generated") 
print("✓ Budget Analysis & Table: Generated")
print("✓ Full-Day Schedule: Generated")
print("✓ Consolidated Final Plan: Generated")
print("-" * 80)

print("\n📊 Expected Plan Sections:")
print("  1. Executive Summary of the event")
print("  2. Hierarchical 3-level breakdown")
print("  3. Vendor & venue recommendations")
print("  4. Comprehensive itemized budget")
print("  5. Master schedule with contingencies")
print("  6. Detailed booking checklist")
print("  7. Risk assessment & mitigation")
print("  8. Complete execution roadmap")


FINAL CONSOLIDATED EVENT PLAN

None

PLANNING PROCESS COMPLETED SUCCESSFULLY

📋 OUTPUT SUMMARY:
--------------------------------------------------------------------------------
✓ Hierarchical Event Plan: Generated
✓ Vendor Research Summary: Generated
✓ Budget Analysis & Table: Generated
✓ Full-Day Schedule: Generated
✓ Consolidated Final Plan: Generated
--------------------------------------------------------------------------------

📊 Expected Plan Sections:
  1. Executive Summary of the event
  2. Hierarchical 3-level breakdown
  3. Vendor & venue recommendations
  4. Comprehensive itemized budget
  5. Master schedule with contingencies
  6. Detailed booking checklist
  7. Risk assessment & mitigation
  8. Complete execution roadmap


## Bonus: Running the System with Custom Event Inputs

You can customize the event parameters and run the system again with different inputs. Uncomment and modify the section below to plan a different event.

In [12]:
"""
OPTIONAL: Uncomment and modify to test with different event inputs

# Example 1: Product Launch Event
custom_inputs_1 = {
    "event_goal": "Plan a product launch event for a new SaaS platform",
    "date": "2025-12-01",
    "attendee_count": "150 attendees",
    "preferences": "Luxury setting, tech-focused audience, live demo capabilities, press coverage"
}

# Example 2: Wedding Event
custom_inputs_2 = {
    "event_goal": "Organize a luxury wedding reception",
    "date": "2025-06-21",
    "attendee_count": "200 guests",
    "preferences": "Elegant black-tie, outdoor venue with garden, vegetarian options, live band"
}

# Example 3: Academic Conference
custom_inputs_3 = {
    "event_goal": "Organize a 2-day academic research conference",
    "date": "2025-09-15",
    "attendee_count": "500 attendees",
    "preferences": "Professional setting, multiple breakout rooms, academic rigor, networking focus"
}

# To run with custom inputs, uncomment below and modify the inputs variable:

# Uncomment to use different inputs:
# custom_inputs = custom_inputs_1  # Change to _2 or _3 for other events
# 
# crew_custom = build_event_planning_crew(
#     event_goal=custom_inputs["event_goal"],
#     date=custom_inputs["date"],
#     attendee_count=custom_inputs["attendee_count"],
#     preferences=custom_inputs["preferences"]
# )
# 
# result_custom = crew_custom.kickoff(inputs=custom_inputs)
# print("\\n" + "="*80)
# print("CUSTOM EVENT PLAN")
# print("="*80 + "\\n")
# print(result_custom)
"""

print("\n💡 Tip: You can run this system with different event types by modifying the inputs in the cell above!")
print("   Try planning a product launch, wedding, conference, or any other event type!")
print("\n✓ Hierarchical Planning System Ready!")
print("  All 5 agents are configured and ready to plan your event.")


💡 Tip: You can run this system with different event types by modifying the inputs in the cell above!
   Try planning a product launch, wedding, conference, or any other event type!

✓ Hierarchical Planning System Ready!
  All 5 agents are configured and ready to plan your event.
